In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import openpyxl
from peft import PeftModel
import pandas as pd

In [ ]:
# Paths
base_model = "Qwen/Qwen2.5-3B-Instruct"
data_path = r"C:\Users\...\val_dataset.json"
lora_path = r"C:\Users\...\peft-128"

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    base_model,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

# Base model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype="auto",
    device_map="auto"
)

# Load LoRA
model = PeftModel.from_pretrained(
    model,
    lora_path
)

model.eval()

c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\accelerate\utils\modeling.py:1462: UserWarning: Current model requires 218105472 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:22<00:00, 11.08s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\accelerate\utils\modeling.py:1462: UserWarning: Current model requires 603988992 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
                (base_la

In [ ]:
# Load Dataset
df = pd.read_json(
    data_path,
    lines=True
)

# Prompt and Inference
predictions = []

for idx, row in df.iterrows():

    tupi_text = row["Tupi"]

    prompt = f"""<|im_start|>system
    Você é um assistente de IA muito útil para traduções.<|im_end|>
    <|im_start|>user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    {tupi_text}<|im_end|>
    <|im_start|>assistant
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            max_length=50,
            #max_tokens=512,
            top_p=0.9,
            temperature=0.1,
            do_sample=True,
            #repetition_penalty=1.05,
            #eos_token_id=tokenizer.eos_token_id,
            #pad_token_id=tokenizer.eos_token_id,
        )


    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    #response = tokenizer.decode(
    #    outputs[0][inputs["input_ids"].shape[1]:],
    #    skip_special_tokens=True
    #).strip()

    #response = response.replace(
    #    "Fim da tradução.",
    #    ""
    #).strip()

    predictions.append({
        "Tupi": tupi_text,
        "Português_esperado": row["Português"],
        "Português_predito": response,
    })

    print("=" * 80)
    print("TUPI:")
    print(tupi_text)

    print("\nESPERADO:")
    print(row["Português"])

    print("\nPREDITO:")
    print(response)

Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TUPI:
Agoacem apyába cetà;

ESPERADO:
Achei muitos índios.

PREDITO:
system
    Você é um assistente de IA muito útil para traduções.
    user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    Agoacem apyába cetà;
    assistant
     Então, aí, a gente vai.


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TUPI:
abá guyra'i oîpsyky ka'ape

ESPERADO:
o homem capturou o passarinho na mata

PREDITO:
system
    Você é um assistente de IA muito útil para traduções.
    user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    abá guyra'i oîpsyky ka'ape
    assistant
     o homem caminhou pela praia.


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

TUPI:
pitanga osó oka rupi 'ybá 'oka

ESPERADO:
a criança foi pela casa para colher frutas

PREDITO:
system
    Você é um assistente de IA muito útil para traduções.
    user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    pitanga osó oka rupi 'ybá 'oka
    assistant
     a criança foi pela casa para colher frutas na mata.


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TUPI:
abá tatu oîpsyky ybytýrape

ESPERADO:
o homem capturou o tatu na serra

PREDITO:
system
    Você é um assistente de IA muito útil para traduções.
    user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    abá tatu oîpsyky ybytýrape
    assistant
     o homem capturou o tatu na serra.


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TUPI:
Akûé gûarani oîamotare'ym ebokûé paîé

ESPERADO:
Aquele guerreiro odeia esse pajé

PREDITO:
system
    Você é um assistente de IA muito útil para traduções.
    user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    Akûé gûarani oîamotare'ym ebokûé paîé
    assistant
    Então, o guerreiro caminhou pela praia comigo.
TUPI:
îagûara osó 'yembyîeîa rupi 'ybá 'oka

ESPERADO:
a onça foi pela praia para colher frutas

PREDITO:
system
    Você é um assistente de IA muito útil para traduções.
    user
    Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

    îagûara osó 'yembyîeîa rupi 'ybá 'oka
    assistant
     a onça foi pela serra para colher frutas na mata.


In [6]:
pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    "predictions.csv",
    index=False
)